In [1]:
# Install deps
!pip uninstall -y sympy transformers sentence-transformers
!pip install sympy==1.12 transformers==4.38.2 sentence-transformers==2.6.1
!pip install pinecone nltk tiktoken -q

# Env config
%env PINECONE_API_KEY=pcsk_5zmtAT_Hf7BPztEgkqyeNs9Lb9kHoihUxmi3ZKT1CN14gSLwPxryDiPmYGDrT9eoyEkR8z
%env INDEX_NAME=knowledge-base-2

Found existing installation: sympy 1.14.0
Uninstalling sympy-1.14.0:
  Successfully uninstalled sympy-1.14.0
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: sentence-transformers 5.2.2
Uninstalling sentence-transformers-5.2.2:
  Successfully uninstalled sentence-transformers-5.2.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 9.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.3/163.3 kB 10.4 MB/s eta 0:00:00
   ━

In [2]:
# Initalise tokenizers


from pinecone import Pinecone, ServerlessSpec
import os, json, re
from typing import Dict, List, Any, Iterable, Tuple

import nltk
from nltk.tokenize import sent_tokenize
import tiktoken

from sentence_transformers import SentenceTransformer

nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [4]:
# Load the glossary
from google.colab import drive
drive.mount('/content/drive')

GLOSSARY_PATH = "/content/drive/MyDrive/Polaris/Knowledge Base 2.0/abbrev_glossary.json"

with open(GLOSSARY_PATH, "r", encoding="utf-8") as f:
    GLOSSARY: Dict[str, str] = json.load(f)

print(f"Loaded glossary entries: {len(GLOSSARY)}")

Mounted at /content/drive
Loaded glossary entries: 98


In [23]:
# Initialise Pinecone connection and create an index if needed
import os

pc = Pinecone(api_key=f"{os.getenv("PINECONE_API_KEY")}")

INDEX_NAME = os.getenv("INDEX_NAME")
DIMENSIONS = 768

if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSIONS,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(INDEX_NAME)
print(index.describe_index_stats())

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '150',
                                    'content-type': 'application/json',
                                    'date': 'Wed, 11 Feb 2026 17:06:54 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '42',
                                    'x-pinecone-request-latency-ms': '42',
                                    'x-pinecone-response-duration-ms': '44'}},
 'dimension': 768,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}


In [7]:
# Load embedding model and create text helper

model = SentenceTransformer("all-mpnet-base-v2")


# Uses glossary to expand abbreviated words for better retrieval [e.g. EP becomes EP (Exchange Participant)]
def expand_abbreviations(text: str, glossary: Dict[str, str]) -> str:
    if not text:
        return text

    keys = sorted(glossary.keys(), key=len, reverse=True)
    for k in keys:
        meaning = glossary[k]
        text = re.sub(rf"\b{re.escape(k)}\b", f"{k} ({meaning})", text)
    return text


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [19]:
# Load the competencies and create metadata

def load_competencies_from_file(path: str) -> List[Dict[str, Any]]:

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, list):
        return data

    raise ValueError(f"Unrecognized JSON format in {path}")


def competency_to_metadata(competency: Dict[str, Any], file_name: str) -> Dict[str, Any]:
    src = competency.get("source") or {}

    skill_behaviors = competency.get("skill_level_behaviors", {})
    role_requirements = competency.get("role_level_requirements", {})

    def _get_roles_for_level(level_name: str) -> List[str]:
      roles = []
      for role, required_level in role_requirements.items():
          normalized_required = required_level.strip().lower()
          if normalized_required == level_name.lower():
              roles.append(expand_abbreviations(role, GLOSSARY))
      return roles

    all_roles = []
    for role, required_level in role_requirements.items():
        if required_level.strip().lower() != "non-existent":
            all_roles.append(expand_abbreviations(role, GLOSSARY))

    min_role = all_roles[0] if all_roles else "Member"
    max_role = all_roles[-1] if all_roles else "Expert"

    normalized_description = expand_abbreviations(competency.get("description", ""), GLOSSARY)

    normalized_keywords = []
    for keyword in competency.get("cv_keywords", []):
        normalized_keywords.append(expand_abbreviations(keyword, GLOSSARY))

    normalized_context = []
    for context in competency.get("aiesec_use_cases", []):
        normalized_context.append(expand_abbreviations(context, GLOSSARY))

    normalized_behaviors = {
        "beginner": expand_abbreviations(skill_behaviors.get("Beginner", ""), GLOSSARY),
        "intermediate": expand_abbreviations(skill_behaviors.get("Intermediate", ""), GLOSSARY),
        "advanced": expand_abbreviations(skill_behaviors.get("Advanced", ""), GLOSSARY),
        "expert": expand_abbreviations(skill_behaviors.get("Expert", ""), GLOSSARY),
    }



    return {
        "function_code": competency.get("function_code", ""),
        "function_name": competency.get("function_name", ""),
        "skill_type": competency.get("skill_type", ""),

        "competency_id": competency.get("competency_id", ""),
        "competency_name": competency.get("competency_name", ""),

        "description": normalized_description,
        "cv_keywords": normalized_keywords,
        "aiesec_context": normalized_context,

        "beginner_behaviour": normalized_behaviors.get("beginner", ""),
        "beginner_roles": _get_roles_for_level("beginner"),

        "intermediate_behaviour": normalized_behaviors.get("intermediate", ""),
        "intermediate_roles": _get_roles_for_level("intermediate"),

        "advanced_behaviour": normalized_behaviors.get("advanced", ""),
        "advanced_roles": _get_roles_for_level("advanced"),

        "expert_behaviour": normalized_behaviors.get("expert", ""),
        "expert_roles": _get_roles_for_level("expert"),

        "all_roles": all_roles,
        "min_role": min_role,
        "max_role": max_role,
        "source_file": file_name
    }

In [26]:
# Ingest competencies, create embedding text and then do a stable upsert into pinecone

COMPETENCY_FOLDER = "/content/drive/MyDrive/Polaris/Knowledge Base 2.0/Competency JSONs"
json_files = [f for f in os.listdir(COMPETENCY_FOLDER) if f.endswith(".json")]
print(f"Found {len(json_files)} files: {json_files}")

def batched(it: List[Any], batch_size: int) -> Iterable[List[Any]]:
    for i in range(0, len(it), batch_size):
        yield it[i:i+batch_size]

def competency_to_embedding_text(metadata: Dict[str, Any]) -> str:
    lines = [
        f"Competency: {metadata['competency_name']}",
        f"Function: {metadata['function_name']} ({metadata['function_code']})",
        f"Type: {metadata['skill_type']} Skill",
        f"Description: {metadata['description']}",
        f"Use Cases: {', '.join(metadata['aiesec_context'])}",
        f"Keywords: {', '.join(metadata['cv_keywords'])}",
        "",
        "Skill Progression - ",
        f"- Beginner, {metadata['beginner_roles']}: {metadata['beginner_behaviour']}",
        f"- Intermediate, {metadata['intermediate_roles']}: {metadata['intermediate_behaviour']}",
        f"- Advanced, {metadata['advanced_roles']}: {metadata['advanced_behaviour']}",
        f"- Expert, {metadata['expert_roles']}: {metadata['expert_behaviour']}",
    ]
    return "\n".join(lines).strip()


vectors_to_upsert = []
total_competencies = 0

for file_name in json_files:
    path = os.path.join(COMPETENCY_FOLDER, file_name)
    competencies = load_competencies_from_file(path)

    for competency in competencies:
        metadata = competency_to_metadata(competency, file_name)
        embedding_text = competency_to_embedding_text(metadata)

        vector = model.encode(embedding_text, convert_to_numpy=True).tolist()
        vector_id = f"{file_name}:{metadata.get("competency_id")}"

        vectors_to_upsert.append({
            "id": vector_id,
            "values": vector,
            "metadata": {
                **metadata,
                "text": embedding_text
            }
        })
        total_competencies += 1

    print(f"Prepared {file_name}: {len(competencies)} competencies")

print(f"\nUpserting {len(vectors_to_upsert)} vectors into '{INDEX_NAME}'...")
for batch in batched(vectors_to_upsert, batch_size=200):
    index.upsert(vectors=batch)

print(f"✅ Done. Upserted {total_competencies} competencies.")
print(index.describe_index_stats())

Found 7 files: ['im_competencies.json', 'fin_competencies.json', 'mgmt_competencies.json', 'mkt_competencies.json', 'ops_competencies.json', 'sales_competencies.json', 'tm_competencies.json']
Prepared im_competencies.json: 23 competencies
Prepared fin_competencies.json: 19 competencies
Prepared mgmt_competencies.json: 21 competencies
Prepared mkt_competencies.json: 28 competencies
Prepared ops_competencies.json: 22 competencies
Prepared sales_competencies.json: 25 competencies
Prepared tm_competencies.json: 20 competencies

Upserting 158 vectors into 'knowledge-base-2'...
✅ Done. Upserted 158 competencies.
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '185',
                                    'content-type': 'application/json',
                                    'date': 'Wed, 11 Feb 2026 17:26:48 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
 

In [27]:
# Retrieval test suite for knowledge-base-2

from typing import Optional

def query_index(query_text: str, top_k: int = 10, filters: Optional[dict] = None):
    query_vector = model.encode(query_text, convert_to_numpy=True).tolist()
    result = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True,
        filter=filters
    )
    return result.get("matches", [])


def print_match(match: dict, verbose: bool = False):
    md = match.get("metadata", {})
    score = match.get("score", 0)

    bar_len = int(score * 20)
    score_bar = "█" * bar_len + "░" * (20 - bar_len)

    print(f"\n{'='*60}")
    print(f"  Score : [{score_bar}] {score:.4f}")
    print(f"  Name  : {md.get('competency_name', '???')} ({md.get('competency_id', '???')})")
    print(f"  Function  : {md.get('function_name', '???')} ({md.get('function_code', '???')})")
    print(f"  Skill Type  : {md.get('skill_type', '???')}")
    print(f"  Role Range : {md.get('min_role', '???')} → {md.get('max_role', '???')}")
    print(f"  Roles : {', '.join(md.get('all_roles', []))}")
    print(f"  Source File  : {md.get('source_file', '???')}")

    print(f"\n  Description:\n    {md.get('description', '')}")
    print(f"\n  Context:\n    {chr(10).join('    - ' + c for c in md.get('aiesec_context', []))}")
    print(f"\n  Keywords: {', '.join(md.get('cv_keywords', []))}")
    print(f"\n  Skill Levels:")
    for level in ["beginner", "intermediate", "advanced", "expert"]:
        behaviour = md.get(f"{level}_behaviour", "")
        roles = md.get(f"{level}_roles", [])
        if behaviour:
            print(f"    [{level.upper():12}] {behaviour}")
            print(f"    {'':14} → Roles: {', '.join(roles) if roles else 'none'}")
    print(f"\n  Embedded Text:")
    print(f"    {md.get("text", "")}")

    # Metadata health checks
    warnings = []
    if not md.get("competency_id"):
        warnings.append("⚠️  Missing competency_id")
    if not md.get("all_roles"):
        warnings.append("⚠️  Empty all_roles")
    if not md.get("beginner_behaviour"):
        warnings.append("⚠️  Missing beginner_behaviour")
    if not md.get("text"):
        warnings.append("⚠️  Missing embedded text")
    if md.get("chunk_index") is not None:
        warnings.append("⚠️  chunk_index still present (old vector?)")
    if warnings:
        print(f"\n  {'  '.join(warnings)}")


def run_retrieval_tests():

    test_cases = [
        {
            "label": "🔍 Semantic match (risk & finance)",
            "query": "Identify financial risks and mitigate losses",
            "filters": None,
            "top_k": 3,
            "verbose": True,
        },
        {
            "label": "🔍 Role-based query (LCVP skills)",
            "query": "What skills does an LCVP need to manage their team?",
            "filters": None,
            "top_k": 3,
            "verbose": False,
        },
        {
            "label": "🔍 Filtered by function (Finance only)",
            "query": "Budget planning and forecasting",
            "filters": {"function_code": {"$eq": "FIN"}},
            "top_k": 3,
            "verbose": False,
        },
        {
            "label": "🔍 Filtered by skill type (Hard skills only)",
            "query": "Tracking expenses and cash flow",
            "filters": {"skill_type": {"$eq": "Hard"}},
            "top_k": 3,
            "verbose": False,
        },
        {
            "label": "🔍 CV / career query",
            "query": "Skills I can put on my CV after managing event budgets",
            "filters": None,
            "top_k": 3,
            "verbose": False,
        },
        {
            "label": "🔍 Abbreviation expansion check (EP, LC, EB)",
            "query": "Exchange Participant reimbursement and Local Committee budget",
            "filters": None,
            "top_k": 3,
            "verbose": False,
        },
    ]

    print("=" * 60)
    print("  POLARIS RETRIEVAL TEST SUITE")
    print(f"  Index : {INDEX_NAME}")
    print(f"  Model : all-mpnet-base-v2")
    print("=" * 60)

    for test in test_cases:
        print(f"\n\n{'#'*60}")
        print(f"  {test['label']}")
        print(f"  Query: \"{test['query']}\"")
        if test["filters"]:
            print(f"  Filter: {test['filters']}")
        print(f"{'#'*60}")

        matches = query_index(test["query"], top_k=test["top_k"], filters=test["filters"])

        if not matches:
            print("  ❌ No matches returned")
            continue

        for match in matches:
            print_match(match, verbose=test["verbose"])

    # Index health summary
    print(f"\n\n{'='*60}")
    print("  INDEX HEALTH SUMMARY")
    print("=" * 60)
    stats = index.describe_index_stats()
    print(f"  Total vectors : {stats['total_vector_count']}")
    print(f"  Dimensions    : {stats['dimension']}")
    print(f"  Namespaces    : {list(stats.get('namespaces', {}).keys()) or ['default']}")


run_retrieval_tests()

  POLARIS RETRIEVAL TEST SUITE
  Index : knowledge-base-2
  Model : all-mpnet-base-v2


############################################################
  🔍 Semantic match (risk & finance)
  Query: "Identify financial risks and mitigate losses"
############################################################

  Score : [██████████░░░░░░░░░░] 0.5032
  Name  : Risk Management (fin_hard_risk_management)
  Function  : Finance (FIN)
  Skill Type  : Hard
  Role Range : Level 1 : Member → Level 4 : AI (AIESEC International)
  Roles : Level 1 : Member, Level 2 : TL (Team Leader), Level 3 : LCVP (Local Committee Vice-President) / EST (Entity Support Team), Level 4 : MCVP (Member Committee Vice-President) / GST (Global Support Team), Level 4 : AI (AIESEC International)
  Source File  : fin_competencies.json

  Description:
    Identifying, assessing, and mitigating financial and operational risks

  Context:
        - Anticipating payment delays
    - planning emergency budget
    - potential financial 